# LLM4Rec: SASRec to LLaMA Adapter Pipeline
Full pipeline for ECS172 project — runs on a Colab T4 GPU.

**Step order:**
1. Mount Drive & set path
2. Install dependencies
3. Train SASRec (Phase 1) → `checkpoints/item_embeddings.pt`
4. Train Adapter (Phase 2) → `checkpoints/adapter.pt`
5. Cache projected embeddings (one-shot, fast)
6. Run all 4 evaluations (text-only baseline + 3 injection modes)

## Cell 1: Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
# IMPORTANT: Change this path to where your ECS172 folder is located in your Google Drive
PROJECT_DIR = '/content/drive/MyDrive/ECS172'
os.chdir(PROJECT_DIR)
print('Current directory:', os.getcwd())
!ls

Current directory: /content/drive/MyDrive/ECS172
checkpoints	      results_ac.json	   test_ranking_prompts_AC.json
Colab_AC_Eval.ipynb   results_mode_A.json  test_ranking_prompts.json
Colab_Pipeline.ipynb  results_mode_C.json  train.csv
md_files	      scripts		   train_ranking_prompts_5cand.json
README.md	      src		   val.csv
requirements.txt      test_ranking.csv	   val_prompts.json


## Cell 2: Install Dependencies

In [3]:
!pip install -q -r requirements.txt
# Unsloth gives ~2x faster LLaMA loading/inference on Colab T4
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
print('Done.')

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 160.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 924.2/924.2 kB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 129.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 127.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 27.4 MB/s eta 0:00:00
Done.


## Cell 3: (Optional) HuggingFace Login
Only needed if you want to use the gated `meta-llama/Llama-3.2-1B-Instruct` model.
Skip this cell if you use the default `unsloth/Llama-3.2-1B-Instruct` (no token required).

In [4]:
# from huggingface_hub import login
# login("YOUR_HF_TOKEN_HERE")

## Cell 4: Train SASRec (Phase 1)
Creates `checkpoints/item_embeddings.pt` and `checkpoints/id_maps.json`.
Only needs to run once. ~5-10 min on T4.

In [5]:
!python scripts/train_sasrec.py --epochs 15

[device] cuda
[data] loading /content/drive/MyDrive/ECS172/train.csv (implicit: rating >= 4.0)
[data] n_items=3510, users=5434
[data] 5433 training sequences
[data] 3198 val ranking samples (1 pos + 100 negs)
[train] SASRec paper protocol | epochs=15 | batch=128 | max_len=200 | d=50 | BCE+1neg
epoch 1/15  train_bce=1.0483  HR@10=0.5094  NDCG@10=0.2828  (n_val=3198, 9.8s)
epoch 2/15  train_bce=0.9032  HR@10=0.5147  NDCG@10=0.2834  (n_val=3198, 8.7s)
epoch 3/15  train_bce=0.8700  HR@10=0.4841  NDCG@10=0.2647  (n_val=3198, 8.7s)
epoch 4/15  train_bce=0.8144  HR@10=0.4822  NDCG@10=0.2629  (n_val=3198, 8.7s)
epoch 5/15  train_bce=0.7732  HR@10=0.4800  NDCG@10=0.2581  (n_val=3198, 8.8s)
epoch 6/15  train_bce=0.7508  HR@10=0.5013  NDCG@10=0.2733  (n_val=3198, 8.7s)
epoch 7/15  train_bce=0.7312  HR@10=0.5219  NDCG@10=0.2871  (n_val=3198, 8.7s)
epoch 8/15  train_bce=0.7093  HR@10=0.5375  NDCG@10=0.3010  (n_val=3198, 8.7s)
epoch 9/15  train_bce=0.6930  HR@10=0.5222  NDCG@10=0.2931  (n_val=3198, 

## Cell 5: Train Adapter (Phase 2)
Trains the MLP projector using LLaMA-grounded teacher-forced CE loss on movie titles.
Creates `checkpoints/adapter.pt`.

**To use your fine-tuned LLaMA:** Change `--llm-model` to `./llama31-1b-movielens-full-final`

In [6]:
!python scripts/train_adapter.py \
    --epochs 10 \
    --batch-size 8 \
    --llm-model unsloth/Llama-3.2-1B-Instruct

[device] cuda (dtype=torch.bfloat16)
[data] 3510 item embeddings, dim=50
[data] 3509 movies with title+genre text
[split] 3159 train movies, 350 val movies
[llm] loading unsloth/Llama-3.2-1B-Instruct (first run downloads ~2-5 min)...
config.json: 100% 894/894 [00:00<00:00, 4.61MB/s]
tokenizer_config.json: 100% 54.7k/54.7k [00:00<00:00, 117MB/s]
tokenizer.json: 100% 17.2M/17.2M [00:01<00:00, 15.0MB/s]
special_tokens_map.json: 100% 454/454 [00:00<00:00, 3.10MB/s]
chat_template.jinja: 100% 3.83k/3.83k [00:00<00:00, 17.3MB/s]
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
model.safetensors: 100% 2.47G/2.47G [00:07<00:00, 314MB/s]
Loa

## Cell 6: Cache Projected Embeddings
Runs the adapter over ALL movies once and saves the results to disk.
This is fast (~seconds) and means the evaluation loop never needs to call the adapter again.
Creates `checkpoints/projected_embeddings.pt` of shape `(n_items, llm_dim)`.

In [7]:
!python scripts/cache_projected_embeddings.py

[data] 3510 items, sasrec_dim=50
[adapter] adapter.pt  (llama_grounded_ce)
[adapter] 50 → 1024 → 2048
[adapter] trained against: unsloth/Llama-3.2-1B-Instruct
[adapter] best_val_CE=2.8934
[done] saved /content/drive/MyDrive/ECS172/checkpoints/projected_embeddings.pt  shape=(3510, 2048)


## Cell 6b: Train Ranking Adapter (Phase 2b)
Retrains the adapter using **listwise ranking CE** (Mode C evaluation objective) instead of
title reconstruction. Each forward pass injects 10 candidate soft tokens and trains the adapter
to assign maximum log-probability to the true candidate letter.

Optionally adds a reconstruction grounding term (`--recon-lambda`) to prevent semantic drift.
Validates each epoch using HR@1 via `rank_by_logprob` -- the exact eval metric -- and saves
the best-HR@1 checkpoint as `checkpoints/adapter_ranking.pt`.

**Training time:** ~25-40 min on T4 for 3 epochs / 4000 examples.


In [5]:
# Phase 2b: Train ranking adapter (warm-starts from adapter.pt, validates by HR@1)
# Saves: checkpoints/adapter_ranking.pt
!python scripts/train_adapter_ranking.py --model unsloth/Llama-3.2-1B-Instruct --epochs 8 --max-train 4000 --recon-lambda 0.3 --val-fraction 0.1


[device] cuda
[config] model=unsloth/Llama-3.2-1B-Instruct  recon_lambda=0.3  chat_template=False
[data] 3509 movie titles loaded for reconstruction grounding
[data] 3600 train  /  400 val ranking examples
config.json: 100% 894/894 [00:00<00:00, 4.67MB/s]
tokenizer_config.json: 100% 54.7k/54.7k [00:00<00:00, 136MB/s]
tokenizer.json: 100% 17.2M/17.2M [00:01<00:00, 15.3MB/s]
special_tokens_map.json: 100% 454/454 [00:00<00:00, 2.80MB/s]
chat_template.jinja: 100% 3.83k/3.83k [00:00<00:00, 11.8MB/s]
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
model.safetensors: 100% 2.47G/2.47G [00:07<00:00, 347MB/s]
Loading weights: 100% 146/146 [

## Cell 6c: Cache Ranking Projected Embeddings
Projects all SASRec embeddings through the **ranking** adapter and saves to
`checkpoints/projected_embeddings_ranking.pt` -- distinct from the reconstruction
cache so the two are never silently mixed up.


In [6]:
# Cache projections from the ranking adapter (separate file from reconstruction cache)
!python scripts/cache_projected_embeddings.py --adapter checkpoints/adapter_ranking.pt --out checkpoints/projected_embeddings_ranking.pt


[data] 3510 items, sasrec_dim=50
[adapter] adapter_ranking.pt  (contrastive_ranking)
[adapter] 50 → 1024 → 2048
[adapter] trained against: unsloth/Llama-3.2-1B-Instruct
[adapter] best_val_HR@1=0.4175
[done] saved checkpoints/projected_embeddings_ranking.pt  shape=(3510, 2048)


## Cell 7: Build A-J Prompts, Sanity Check, Run Mode A vs Mode C

In [7]:
# Cell 7: Build prompts (A-J) + pre-flight sanity, then run Mode A vs Mode C
# Uses the RANKING adapter cache (projected_embeddings_ranking.pt)
LLM_MODEL = 'unsloth/Llama-3.2-1B-Instruct'

# 1) Materialise the letter-based prompt artifact (torch-free, verifies off-by-one)
!python scripts/build_AC_prompts.py

# 2) Pre-flight: cache round-trip + single-token letters (catches silent failures)
!python scripts/sanity_check.py --model {LLM_MODEL}

# 3) Evaluate Mode A (text) and Mode C (ranking adapter soft tokens)
#    --projected-embeddings points at the RANKING cache, not the reconstruction one
!python scripts/eval_ranking.py --modes A C --model {LLM_MODEL} --projected-embeddings checkpoints/projected_embeddings_ranking.pt --embedding-adapter checkpoints/adapter_ranking.pt --n-candidates 10


[done] wrote 604 rows -> /content/drive/MyDrive/ECS172/test_ranking_prompts_AC.json
[ok] all rows: candidates[true_positive_pos] == true_positive_id (no off-by-one)

--- sample prompt_A (user 5 ) ---
A user has rated the following movies:
- Henry: Portrait of a Serial Killer (1990) (Crime|Horror): 1/5
- Joe Gould's Secret (2000) (Drama): 2/5
- Wings of the Dove, The (1997) (Drama|Romance|Thriller): 3/5
- Speed (1994) (Action|Romance|Thriller): 4/5
- Thin Red Line, The (1998) (Action|Drama|War): 5/5
- Much Ado About Nothing (1993) (Comedy|Romance): 3/5
- Opposite of Sex, The (1998) (Comedy|Drama): 4/5
- Celluloid Closet, The (1995) (Documentary): 3/5
- Being John Malkovich (1999) (Comedy): 5/5
- Living in Oblivion (1995) (Comedy): 4/5

From the list below, rank which movie this user would most likely enjoy:
A. Mutters Courage (1995) (Comedy)
B. Hard-Boiled (Lashou shentan) (1992) (Action|Crime)
C. Bram Stoker's Dracula (1992) (Horror|Romance)
D. League of Their Own, A (1992) (Comedy|Dra

## Cell 8: Compare Results (A vs C + random baseline)

In [8]:
# Cell 8: Compare Mode A vs Mode C (with random baseline)
import json
from pathlib import Path

conditions = [
    ('Mode A (text baseline)',         'results_mode_A.json'),
    ('Mode C (candidate soft tokens)', 'results_mode_C.json'),
]

ks, rand = [], {}
for _, f in conditions:
    pth = Path(f)
    if pth.exists():
        data = json.loads(pth.read_text())
        ks = sorted(int(k.split('@')[1]) for k in data['metrics'] if k.startswith('HR@'))
        rand = data.get('random_baseline', {})
        break

def fmt_row(label, m):
    cells = ''.join(f"{m.get(f'HR@{k}', 0):>8.3f}" for k in ks)
    cells += ''.join(f"{m.get(f'NDCG@{k}', 0):>9.4f}" for k in ks)
    return f"{label:<32}{cells}"

header = f"{'Condition':<32}" + ''.join(f"{'HR@'+str(k):>8}" for k in ks) + ''.join(f"{'NDCG@'+str(k):>9}" for k in ks)
print(header)
print('-' * len(header))
if rand:
    print(fmt_row('Random', rand))
for label, fname in conditions:
    pth = Path(fname)
    if not pth.exists():
        print(f"{label:<32}  (not run yet)")
        continue
    print(fmt_row(label, json.loads(pth.read_text())['metrics']))


Condition                           HR@1    HR@3    HR@5   NDCG@1   NDCG@3   NDCG@5
-----------------------------------------------------------------------------------
Random                             0.088   0.280   0.497   0.0877   0.1957   0.2842
Mode A (text baseline)             0.126   0.374   0.556   0.1258   0.2632   0.3376
Mode C (candidate soft tokens)     0.320   0.631   0.791   0.3195   0.4988   0.5651


In [5]:
# Cell 8: Compare Mode A vs Mode C + letter probability analysis
import json
from pathlib import Path
from collections import defaultdict

CONDITIONS = [
    ('Mode A (text baseline)',         'results_mode_A.json'),
    ('Mode C (candidate soft tokens)', 'results_mode_C.json'),
]
LETTERS = 'ABCDEFGHIJ'

# ── Load all results ──────────────────────────────────────────────
results = {}
for label, fname in CONDITIONS:
    p = Path(fname)
    if p.exists():
        results[label] = json.loads(p.read_text())

if not results:
    print('No result files found. Run Cell 7 first.')
else:

    # ── Metrics table ─────────────────────────────────────────────────
    first = next(iter(results.values()))
    ks = sorted(int(k.split('@')[1]) for k in first['metrics'] if k.startswith('HR@'))
    rand = first.get('random_baseline', {})

    def fmt_row(label, m):
        hr   = ''.join(f"{m.get(f'HR@{k}',   0):>8.3f}" for k in ks)
        ndcg = ''.join(f"{m.get(f'NDCG@{k}', 0):>9.4f}" for k in ks)
        return f"{label:<32}{hr}{ndcg}"

    hdr = f"{'Condition':<32}" + ''.join(f"{'HR@'+str(k):>8}" for k in ks) + ''.join(f"{'NDCG@'+str(k):>9}" for k in ks)
    print(hdr)
    print('-' * len(hdr))
    if rand:
        print(fmt_row('Random', rand))
    for label, _ in CONDITIONS:
        if label in results:
            print(fmt_row(label, results[label]['metrics']))
        else:
            print(f"{label:<32}  (not run)")

    # ── Per-letter analysis ───────────────────────────────────────────
    print()
    for label, _ in CONDITIONS:
        if label not in results:
            continue
        preds = results[label]['predictions']
        n = len(preds)

        # Average probability assigned to each letter position
        avg_prob  = defaultdict(float)  # mean P(letter) across all examples
        pred_freq = defaultdict(int)    # how often this letter is top prediction
        true_freq = defaultdict(int)    # how often true item is at this position
        correct_by_letter = defaultdict(lambda: [0, 0])  # [correct, total] when true=letter

        for p in preds:
            for letter, prob in p['probs'].items():
                avg_prob[letter] += prob
            pred_freq[p['pred_letter']] += 1
            true_freq[p['true_letter']] += 1
            correct_by_letter[p['true_letter']][1] += 1
            if p['correct']:
                correct_by_letter[p['true_letter']][0] += 1

        letters_used = sorted(avg_prob.keys())

        print(f"{'='*60}")
        print(f"{label}  (n={n})")
        print(f"{'='*60}")
        print(f"{'Letter':>8} {'AvgP':>7} {'PredFreq%':>10} {'TrueFreq%':>10} {'HR@1_cond%':>11}")
        print(f"{'-'*8:>8} {'-'*7:>7} {'-'*10:>10} {'-'*10:>10} {'-'*11:>11}")
        for letter in letters_used:
            ap   = avg_prob[letter] / n
            pf   = 100 * pred_freq[letter] / n
            tf   = 100 * true_freq[letter] / n
            c, t = correct_by_letter[letter]
            hr1c = 100 * c / t if t > 0 else 0.0
            print(f"{letter:>8} {ap:>7.4f} {pf:>10.1f} {tf:>10.1f} {hr1c:>11.1f}")

        # Position bias summary: ideal model has PredFreq% ≈ TrueFreq% ≈ 10% each
        max_pred_letter = max(pred_freq, key=pred_freq.get)
        max_pred_pct    = 100 * pred_freq[max_pred_letter] / n
        print(f"Prediction concentration: {max_pred_letter} chosen {max_pred_pct:.1f}% of the time")
        print(f"  Mean P(top-1 letter): {sum(p['probs'][p['pred_letter']] for p in preds)/n:.4f}")
        print(f"  Overall HR@1: {sum(p['correct'] for p in preds)/n:.4f}")

    # ── Sample predictions (first 5) ─────────────────────────────────
    print()
    print('='*60)
    print('Sample predictions (first 5 users, all modes)')
    print('='*60)
    for label, _ in CONDITIONS:
        if label not in results:
            continue
        print(f"{label} ---")
        for p in results[label]['predictions'][:5]:
            prob_str = ' '.join(f"{l}:{v:.3f}" for l, v in sorted(p['probs'].items()))
            print(f"  user {p['user_id']:>5}: pred={p['pred_letter']} true={p['true_letter']} correct={str(p['correct']):<5}  [{prob_str}]")


Condition                           HR@1    HR@3    HR@5   NDCG@1   NDCG@3   NDCG@5
-----------------------------------------------------------------------------------
Random                             0.088   0.280   0.497   0.0877   0.1957   0.2842
Mode A (text baseline)             0.126   0.374   0.556   0.1258   0.2632   0.3376
Mode C (candidate soft tokens)     0.320   0.631   0.791   0.3195   0.4988   0.5651

Mode A (text baseline)  (n=604)
  Letter    AvgP  PredFreq%  TrueFreq%  HR@1_cond%
-------- ------- ---------- ---------- -----------
       A  0.1922       66.6       11.1        74.6
       B  0.0879        0.5        7.9         0.0
       C  0.1101        2.8       10.9         6.1
       D  0.0646        0.0       10.1         0.0
       E  0.0831        0.3        8.8         0.0
       F  0.0676        0.2       11.1         0.0
       G  0.1073        3.0        9.6         5.2
       H  0.0447        0.0       10.9         0.0
       I  0.0728        0.7        9.